<a href="https://colab.research.google.com/github/sanchezSanjar/jaydariGPT/blob/develop/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
!pip install transformers torch bitsandbytes datasets peft trl

In [32]:
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM, TrainingArguments
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

In [33]:
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print('Vocab size:', tokenizer.vocab_size)
# print('Special tokens:', tokenizer.special_tokens_map)

# quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4'
)

bnb_config

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto', #GPU, CPU
    # dtype=torch.bfloat16
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [34]:
# Before Fine-tuning
prompt="Explain what a tokenizer is."
# prompt='A tokenizer is a tool in natural language processing that'

inputs = tokenizer(prompt, return_tensors='pt').to(model.device) # GPU, CPU

with torch.no_grad():
  output_ids = model.generate(
      **inputs,
      max_new_tokens=80,
      do_sample=True,
      temperature=0.7
  )

  print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

  # print(model)
  first_block = model.model.layers[0]
  print('first_block:', first_block)
  print('=========')
  print(first_block.self_attn)
  print('=========')
  print(model.config)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain what a tokenizer is.
first_block: LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)
LlamaAttention(
  (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
  (v_proj): Linear4bit(in_features=2048, out_feat

In [35]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including frozen 4-bit): {total_params:,}")

Total parameters (including frozen 4-bit): 615,606,272


## datasets library | load_dataset
* instruction tuning

In [36]:
dataset = load_dataset('yahma/alpaca-cleaned', split='train')
dataset
dataset[1]

{'output': 'The three primary colors are red, blue, and yellow. These colors are called primary because they cannot be created by mixing other colors and all other colors can be made by combining them in various proportions. In the additive color system, used for light, the primary colors are red, green, and blue (RGB).',
 'input': '',
 'instruction': 'What are the three primary colors?'}

In [37]:
def generate_prompt(example):
  instruction = example['instruction']
  input_text = example['input']
  output_text = example['output']

  if input_text:
    return(
        '### Instruction:\n'
        f'{instruction}\n\n'
        '### Input:\n'
        f'{input_text}\n\n'
        '### Response:\n'
        f'{output_text}'
    )
  else:
    return(
        '### Instruction:\n'
        f'{instruction}\n\n'
        '### Response:\n'
        f'{output_text}'
    )

def formatting_func(example):
  return {'text': generate_prompt(example)}

dataset = dataset.map(formatting_func)

In [38]:
dataset[0]['text']

'### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.'

In [39]:
dataset = dataset.select(range(7000))

In [40]:
dataset = dataset.shuffle(seed=42)

In [41]:
# [7, 3, 2, 8, 5, 6, 9, 4, 0, 1]
# [7, 3, 2, 8, 5, 6, 9, 4, 0, 1] EXACT SAME ORDER

In [42]:
# [0 , 1, 2, 3, 4, 5, 6, 7, 8, 9]
# [10, 231, 342, 3453, 3464, 5123, 6456, 7, 8, 9]

In [43]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 7000
})

In [44]:
# full Fine-tuning =>
# Cheap Fine-tuning =>
# PEFT => Parameter Efficient Fine Tuning

In [45]:
lora_config = LoraConfig(
    r=8, #rank
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules={'q_proj', 'v_proj'}
)

In [46]:
model = get_peft_model(model, lora_config)

In [47]:
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [48]:
#Qlora
#LoRa

In [53]:
training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4, #VRAM
    num_train_epochs=2, #overfit
    logging_steps=20,
    output_dir='./jaydari_gpt',
    save_strategy='epoch',
    bf16=True,
    fp16=False,
    report_to='none'
)

In [50]:
print(dataset.column_names)

['output', 'input', 'instruction', 'text']


In [55]:
#SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    formatting_func=lambda x: x['text'],
    args=training_args
)

trainer.train()
model.save_pretrained('jaydari_gpt')
tokenizer.save_pretrained('jaydari_gpt')


Step,Training Loss
20,1.370105
40,1.382858
60,1.228485
80,1.315238
100,1.281906
120,1.303799
140,1.330719
160,1.230525
180,1.288237
200,1.194225


('jaydari_gpt/tokenizer_config.json',
 'jaydari_gpt/chat_template.jinja',
 'jaydari_gpt/tokenizer.json')

In [56]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_base = AutoTokenizer.from_pretrained(base_model_id)
model_base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto"
)

model_base.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [57]:
prompt = "Explain what machine learning is in simple words."

inputs_base = tokenizer_base(prompt, return_tensors="pt").to(model_base.device)

with torch.no_grad():
    output_base = model_base.generate(
        **inputs_base,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True
    )

print("===== BASE MODEL OUTPUT =====")
print(tokenizer_base.decode(output_base[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== BASE MODEL OUTPUT =====
Explain what machine learning is in simple words.


##FINE-TUNED MODEL INFERENCE

In [60]:
!pip install --upgrade torchao

model_path = 'jaydari_gpt'

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map='auto'
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 12.0 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [63]:
prompt = """### Instruction:
Explain what machine learning is in simple words.

### Response:
"""

inputs_ft = tokenizer(prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
  output_ft = model.generate(
      **inputs_ft,
      max_new_tokens=120,
      temperature=0.7,
      do_sample=True
  )

  print("==== FINE-TUNED MODEL OUTPUT ====")
  print(tokenizer.decode(output_ft[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


==== FINE-TUNED MODEL OUTPUT ====
### Instruction:
Explain what machine learning is in simple words.

### Response:
Machine learning is the process of designing algorithms to make predictions or make decisions based on historical data. It involves the use of computer algorithms to learn about patterns and relationships in data and to make accurate predictions or recommendations in real-time. Machine learning algorithms are based on statistical models and data analysis techniques, and they can be used to improve the accuracy and efficiency of various applications, such as fraud detection, customer segmentation, navigation, and recommendation systems.


In [64]:
!zip -r jaydari_gpt.zip jaydari_gpt

  adding: jaydari_gpt/ (stored 0%)
  adding: jaydari_gpt/README.md (deflated 44%)
  adding: jaydari_gpt/checkpoint-1750/ (stored 0%)
  adding: jaydari_gpt/checkpoint-1750/README.md (deflated 65%)
  adding: jaydari_gpt/checkpoint-1750/scheduler.pt (deflated 61%)
  adding: jaydari_gpt/checkpoint-1750/trainer_state.json (deflated 80%)
  adding: jaydari_gpt/checkpoint-1750/chat_template.jinja (deflated 60%)
  adding: jaydari_gpt/checkpoint-1750/adapter_config.json (deflated 60%)
  adding: jaydari_gpt/checkpoint-1750/rng_state.pth (deflated 26%)
  adding: jaydari_gpt/checkpoint-1750/training_args.bin (deflated 53%)
  adding: jaydari_gpt/checkpoint-1750/adapter_model.safetensors (deflated 23%)
  adding: jaydari_gpt/checkpoint-1750/tokenizer.json (deflated 85%)
  adding: jaydari_gpt/checkpoint-1750/tokenizer_config.json (deflated 46%)
  adding: jaydari_gpt/checkpoint-1750/optimizer.pt (deflated 22%)
  adding: jaydari_gpt/chat_template.jinja (deflated 60%)
  adding: jaydari_gpt/adapter_config.

In [65]:
from google.colab import files
files.download('jaydari_gpt.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [66]:
#############################
#       MODEL TESTING       #
#############################